# Axial v3 — Iteración A: auditoría de `raw_0`

Auditoría estructural sobre **train/validation** y auditoría probabilística **validation-only**.

- No accede al split test.
- No interpreta `raw_0` como anatomía.
- Usa el checkpoint `axial-final-v2`.
- Agrega soporte temporal y explícito para archivos DICOM `.ima`/`.dcm`.


In [ ]:
# 1) Dependencias
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "pydicom": "pydicom",
}

missing = [
    pip_package
    for module_name, pip_package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *missing,
    ])

print({"installedOrAvailable": list(REQUIRED_PACKAGES.values())})


In [ ]:
# 2) Montar Google Drive
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if not IN_COLAB:
    raise RuntimeError("Este notebook está preparado para ejecutarse en Google Colab.")

from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
MY_DRIVE = DRIVE_MOUNT / "MyDrive"
PFI_ROOT = MY_DRIVE / "PFI_MVP"

if not MY_DRIVE.exists():
    drive.mount(str(DRIVE_MOUNT), force_remount=False)
else:
    print("Google Drive ya está montado.")

if not PFI_ROOT.is_dir():
    raise FileNotFoundError(f"No existe el directorio esperado: {PFI_ROOT}")

print({
    "inColab": IN_COLAB,
    "pfiRoot": str(PFI_ROOT),
    "pfiRootExists": PFI_ROOT.is_dir(),
})


In [ ]:
# 3) Configuración reproducible
import os
from pathlib import Path

REPO_REF = "2b8b943df73bd4a2f45ba4865158301d2d3889cf"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")
REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"

MANIFEST_PATH = (
    PFI_ROOT
    / "results"
    / "E9_alkafri_axial_t2_final_labels_baseline"
    / "E9_t2_final_labels_curated_split.csv"
)

CHECKPOINT_PATH = (
    PFI_ROOT
    / "outputs"
    / "axial_final_v2_training"
    / "resume"
    / "axial-final-v2"
    / "axial_t2_alkafri_v2.best_checkpoint.pt"
)

OUTPUT_ROOT = PFI_ROOT / "outputs"
RUN_ID = "axial-v3-iteration-a-v2-audit-rerun2"

os.environ["PFI_REPO_REF"] = REPO_REF
os.environ["PFI_REPO_ROOT"] = str(REPO_ROOT)
os.environ["PFI_REPO_URL"] = REPO_URL
os.environ["PFI_RUN_AXIAL_V3_AUDIT"] = "1"
os.environ["PFI_DATASET_ROOT"] = str(PFI_ROOT)
os.environ["AXIAL_E9_CURATED_SPLIT_CSV"] = str(MANIFEST_PATH)
os.environ["AXIAL_V2_CHECKPOINT_PATH"] = str(CHECKPOINT_PATH)
os.environ["PFI_OUTPUT_ROOT"] = str(OUTPUT_ROOT)
os.environ["PFI_RUN_ID"] = RUN_ID
os.environ["PFI_MASK_LABEL_MODE"] = "raw"
os.environ["AXIAL_EXPECTED_V2_RUN_ID"] = "axial-final-v2"
os.environ["AXIAL_EXPECTED_AI_SERVICE_COMMIT"] = "285159"
os.environ["PFI_ALLOW_INCOMPLETE_V2_CHECKPOINT_METADATA"] = "0"

print({
    "repoRef": REPO_REF,
    "manifestExists": MANIFEST_PATH.is_file(),
    "manifest": str(MANIFEST_PATH),
    "checkpointExists": CHECKPOINT_PATH.is_file(),
    "checkpoint": str(CHECKPOINT_PATH),
    "outputRoot": str(OUTPUT_ROOT),
    "runId": RUN_ID,
})

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(MANIFEST_PATH)

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(CHECKPOINT_PATH)


In [ ]:
# 4) Clonar o actualizar el repositorio y fijar el commit
import subprocess
import sys

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git",
        "clone",
        REPO_URL,
        str(REPO_ROOT),
    ])

subprocess.check_call([
    "git",
    "-C",
    str(REPO_ROOT),
    "fetch",
    "--all",
    "--tags",
])

subprocess.check_call([
    "git",
    "-C",
    str(REPO_ROOT),
    "checkout",
    REPO_REF,
])

effective_sha = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO_ROOT),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

if effective_sha != REPO_REF:
    raise RuntimeError(
        f"Commit efectivo inesperado: {effective_sha} != {REPO_REF}"
    )

SRC_ROOT = REPO_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

os.environ["PFI_REPO_ROOT"] = str(REPO_ROOT)

print({
    "repoRoot": str(REPO_ROOT),
    "effectiveSha": effective_sha,
    "srcRoot": str(SRC_ROOT),
})


In [ ]:
# 5) Importar módulos y definir el loader médico
import importlib
from pathlib import Path

import numpy as np
import pydicom
from PIL import Image

import lumbar_mri.axial_v3.training as training
import lumbar_mri.axial_v3.iteration_a as iteration_a

# Recargar una sola vez DESPUÉS de configurar entorno y checkout.
training = importlib.reload(training)
iteration_a = importlib.reload(iteration_a)

STANDARD_IMAGE_SUFFIXES = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}


def open_2d_array_medical(path: Path) -> np.ndarray:
    """Abre arrays 2D con compatibilidad DICOM igual a axial-final-v2."""

    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".npy":
        array = np.asarray(np.load(path))

    elif suffix in STANDARD_IMAGE_SUFFIXES:
        array = np.asarray(Image.open(path))

    elif suffix in {".dcm", ".ima"}:
        dicom = pydicom.dcmread(str(path))
        array = np.asarray(dicom.pixel_array)

    else:
        raise ValueError(
            f"Formato médico no soportado: {suffix} ({path})"
        )

    array = np.asarray(array)

    if array.ndim == 2:
        return array

    if array.ndim == 3 and 1 in array.shape:
        squeezed = np.squeeze(array)
        if squeezed.ndim == 2:
            return squeezed

    if (
        suffix in STANDARD_IMAGE_SUFFIXES
        and array.ndim == 3
        and array.shape[-1] in {3, 4}
    ):
        return array[..., 0]

    raise ValueError(
        f"Shape 2D no soportado para {path}: {array.shape}"
    )


# iteration_a importó open_2d_array por nombre desde training,
# por eso deben reemplazarse ambos símbolos.
training.open_2d_array = open_2d_array_medical
iteration_a.open_2d_array = open_2d_array_medical

print({
    "trainingLoader": training.open_2d_array.__name__,
    "iterationALoader": iteration_a.open_2d_array.__name__,
})


In [ ]:
# 6) Crear configuración después de aplicar el loader
CFG = iteration_a.AxialV3AuditConfig()

print({
    "runId": CFG.RUN_ID,
    "manifest": str(CFG.SPLIT_MANIFEST_PATH),
    "checkpoint": str(CFG.V2_CHECKPOINT_PATH),
    "outputRoot": str(CFG.OUTPUT_ROOT),
    "maskLabelMode": CFG.MASK_LABEL_MODE,
    "loaderModule": iteration_a.open_2d_array.__module__,
    "loaderName": iteration_a.open_2d_array.__name__,
})

if iteration_a.open_2d_array is not open_2d_array_medical:
    raise RuntimeError("El loader médico no quedó aplicado en iteration_a.")


In [ ]:
# 7) Preflight del manifest y prueba real de un archivo .ima
import pandas as pd

manifest_df = pd.read_csv(CFG.SPLIT_MANIFEST_PATH)

print({
    "manifestRows": len(manifest_df),
    "splitCounts": (
        manifest_df["split"]
        .astype(str)
        .str.lower()
        .value_counts()
        .to_dict()
    ),
})

records = iteration_a.build_records(CFG)

sample_dicom_path = next(
    Path(record.image_path)
    for record in records
    if Path(record.image_path).suffix.lower() in {".ima", ".dcm"}
)

sample_array = iteration_a.open_2d_array(sample_dicom_path)

print({
    "developmentRecords": len(records),
    "sampleDicom": str(sample_dicom_path),
    "sampleShape": list(sample_array.shape),
    "sampleDtype": str(sample_array.dtype),
    "sampleFinite": bool(np.isfinite(sample_array).all()),
})

if sample_array.ndim != 2:
    raise RuntimeError(f"El DICOM de prueba no es 2D: {sample_array.shape}")


In [ ]:
# 8) Ejecutar la Iteración A completa
# IMPORTANTE: ejecutar CFG, que ya contiene el RUN_ID correcto.
import json

result = iteration_a.run_iteration_a(CFG)

print(json.dumps(result, indent=2, default=str))


In [ ]:
# 9) Verificación final de outputs
import json
from pathlib import Path

RUN_OUTPUT = CFG.OUTPUT_ROOT / CFG.RUN_ID

required_outputs = [
    RUN_OUTPUT / "tables" / "raw0_slice_audit.csv",
    RUN_OUTPUT / "tables" / "raw0_patient_audit.csv",
    RUN_OUTPUT / "tables" / "raw0_validation_probability_audit.csv",
    RUN_OUTPUT / "tables" / "raw0_threshold_margin_grid.csv",
    RUN_OUTPUT / "metrics" / "raw0_audit_summary.json",
    RUN_OUTPUT / "metrics" / "raw0_validation_probability_summary.json",
    RUN_OUTPUT / "reports" / "iteration_a_report.json",
    RUN_OUTPUT / "reports" / "iteration_a_report.md",
    RUN_OUTPUT / "manifests" / "iteration_a_artifacts.json",
]

status = {
    str(path.relative_to(RUN_OUTPUT)): path.is_file()
    for path in required_outputs
}

probability_summary_path = (
    RUN_OUTPUT
    / "metrics"
    / "raw0_validation_probability_summary.json"
)

probability_summary = (
    json.loads(probability_summary_path.read_text(encoding="utf-8"))
    if probability_summary_path.is_file()
    else {}
)

print({
    "runOutput": str(RUN_OUTPUT),
    "allRequiredOutputsExist": all(status.values()),
    "outputs": status,
    "probabilityStatus": probability_summary.get("status"),
    "validationSlices": probability_summary.get("validationSlices"),
})

if not all(status.values()):
    missing_outputs = [
        relative_path
        for relative_path, exists in status.items()
        if not exists
    ]
    raise RuntimeError(
        f"Faltan outputs esperados: {missing_outputs}"
    )

if probability_summary.get("status") != "completed":
    raise RuntimeError(
        "La auditoría probabilística no terminó con status=completed."
    )
